# New-road candidates: why the Java Road link was chosen

This notebook redraws the bar chart of the 12 best new-road candidates, ranked by `scenarios/city/build_planning.py`.

**The rule (report section 3.3):** among pairs of junctions 20–200 m apart with no road between them and no traffic light at either end, add a straight two-way road at 50 km/h and score it by how much it cuts the total empty-road travel time over all junction pairs. The biggest cut wins.

**Input:** one file, `bugis_all_newlink.meta.json`. `build_planning.py` writes it into `scenarios/city/` together with the variant networks; its `ranking` holds the top 12 candidates and their scores.

**How to use it (Colab):** run the cells from top to bottom. If the file is not found, the first code cell opens an upload dialog. No SUMO is needed.

In [ ]:
# --- Find the input file (or upload it in Colab) ----------------------------------------
import json, os

META_FILE = "bugis_all_newlink.meta.json"
CANDIDATES = [META_FILE, os.path.join("scenarios", "city", META_FILE),
              os.path.join("repo", "scenarios", "city", META_FILE)]   # current folder, repository root, or a clone in ./repo

META_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if META_PATH is None:
    try:                                   # in Colab: ask for the file
        from google.colab import files
        print("Please upload", META_FILE)
        files.upload()
        META_PATH = META_FILE if os.path.exists(META_FILE) else None
    except ImportError:                    # not in Colab
        pass
if META_PATH is None:
    raise FileNotFoundError(f"{META_FILE} not found: put it next to this notebook or run build_planning.py first")

meta = json.load(open(META_PATH))
print("Read", META_PATH, "-", len(meta["ranking"]), "ranked candidates")
print("Baseline total empty-road travel time:", f"{meta['baseline_all_pairs_free_flow_time_s']:,.0f} s")

## The ranked candidates as a table

Each row is one candidate new road. `cut (%)` is its score: how much the total empty-road travel time over all junction pairs would fall if it were built. `time now (s)` is today's quickest empty-road time between its two ends, in each direction.

In [ ]:
# --- Short names for the two ends of each candidate ---------------------------------------
import pandas as pd

# Junctions on the Republic Avenue / Nicoll Highway ramps (the network's south-east edge)
RAMPS = {"2015359629": "Republic Ave ramp", "2015359560": "Republic Ave slip road", "2015359296": "Nicoll Hwy ramp"}
# Readable names for the two candidates elsewhere (their junctions join several streets)
SHORT = {"574458676": "Dorset Rd", "cluster_13503436872_4737040981": "Farrer Park Rd",
         "3786038513": "Tekka Ln", "410926262": "Mayne Rd"}

def end_name(r, k):
    """Name of one end ('from' or 'to') of candidate r: a ramp, a short name, or its street names."""
    n, streets = r[k], r[k + "_at"]["streets"]
    return RAMPS.get(n) or SHORT.get(n) or " / ".join(s.replace(" Road", " Rd").replace(" Street", " St") for s in streets)

rows = []
for i, r in enumerate(meta["ranking"]):
    at_ramps = r["from"] in RAMPS or r["to"] in RAMPS
    rows.append({"rank": i + 1,
                 "candidate": f"{end_name(r, 'from')} – {end_name(r, 'to')}",
                 "length (m)": round(r["length_m"]),
                 "cut (%)": round(100 * r["gain_share"], 3),
                 "cut (s)": round(r["gain_s"]),
                 "time now (s)": " / ".join(f"{t:.1f}" for t in r["free_flow_time_now_s"]),
                 "where": "chosen" if i == 0 else ("same ramps" if at_ramps else "elsewhere")})
table = pd.DataFrame(rows).set_index("rank")
n_ramps = int((table["where"] != "elsewhere").sum())
display(table)
print(f"{n_ramps} of the {len(table)} candidates join the Republic Ave / Nicoll Hwy ramps.")

## The chart

`draw(slide=False)` gives the report version with a title, as in the report; `draw(slide=True)` gives the version without a title used on the slide. Both are also saved as PNG files in the current folder.

In [ ]:
# --- Bar chart of the candidates -----------------------------------------------------------
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

COL = {"chosen": "#B4501C", "same ramps": "#2F6DB5", "elsewhere": "#2E9468"}   # orange, blue, green
INK, MUTED, GRID = "#14213D", "#5E6878", "#E6E4DE"

def draw(slide=False, out=None):
    t = table.reset_index()
    fig, ax = plt.subplots(figsize=(9, 5.4) if slide else (11, 6.6), dpi=240 if slide else 200)
    fig.subplots_adjust(left=0.47 if slide else 0.39, right=0.97, top=0.90 if slide else 0.80, bottom=0.11)
    y = list(range(len(t)))[::-1]                                   # rank 1 at the top
    ax.barh(y, t["cut (%)"], height=0.72, color=[COL[w] for w in t["where"]],
            edgecolor="white", linewidth=1.5, zorder=2)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['rank']}.  {r['candidate']}, {r['length (m)']} m" for _, r in t.iterrows()],
                       fontsize=11.5, color=INK)
    ax.get_yticklabels()[0].set_fontweight("bold")
    for (_, r), yy in zip(t.head(2).iterrows(), y[:2]):              # values on the first two bars only
        ax.text(r["cut (%)"] + 0.012, yy, f"{r['cut (%)']:.2f}%", va="center", fontsize=12, color=INK,
                fontweight="bold" if r["where"] == "chosen" else "normal")
    ax.set_xlim(0, 1.05); ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xticklabels([f"{v:.1f}%" for v in ax.get_xticks()], fontsize=11, color=MUTED)
    ax.set_xlabel("Cut in total empty-road travel time over all junction pairs", fontsize=12, color=MUTED)
    ax.grid(axis="x", color=GRID, lw=1, zorder=0)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#C9CCD3"); ax.tick_params(length=0)
    if not slide:
        fig.text(0.02, 0.95, "The Java Road link (Republic Ave ramp – Beach Rd) cuts travel time the most",
                 fontsize=16, fontweight="bold", color=INK)
        fig.text(0.02, 0.905, f"Top {len(t)} of all candidate new roads, as ranked by build_planning.py. "
                 f"{n_ramps} of the {len(t)} join the same ramps at the network's edge to Beach Road.",
                 fontsize=11.5, color=MUTED)
    fig.legend(handles=[Patch(color=COL["chosen"], label="chosen (Java Road link)"),
                        Patch(color=COL["same ramps"], label="same ramps, edge of network"),
                        Patch(color=COL["elsewhere"], label="elsewhere")],
               loc="upper left", bbox_to_anchor=(0.015, 0.985 if slide else 0.885), ncol=3,
               fontsize=11.5, frameon=False, handlelength=1.4)
    if out:
        fig.savefig(out, facecolor="white")
        print("saved", out)
    plt.show()

draw(slide=False, out="newlink_candidates.png")

In [ ]:
# The slide version (no title)
draw(slide=True, out="newlink_candidates_slide.png")

## How the score is computed (for reference)

For every ordered pair of junctions *i*, *j*, let T(i, j) be the quickest empty-road travel time (length ÷ speed limit along the route). A new road between junctions *a* and *b* taking time *t* changes each trip to

T_new(i, j) = min( T(i, j), T(i, a) + t + T(b, j), T(i, b) + t + T(a, j) )

and the score is the fall in the total, Σ [T(i, j) − T_new(i, j)], shown above as a share of today's total. `build_planning.py` computes this for every candidate; this notebook only reads its ranking.